In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone - 4

In [2]:
!pip install -q transformers
!pip install -q peft
!pip install -q accelerate
!pip install -q datasets
!pip install -q sentencepiece

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)

### Question -1

In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

LABEL2ID = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train["label"] = train["answer"].map(LABEL2ID)

print(train.head())

print("\nQ1 Answer")
print(train.loc[150,"label"])

   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believes that humans do not e...   
4  Simultaneity is rel

### Question - 2

In [4]:
row = train.iloc[0]

formatted = str(row["prompt"]) + " [SEP] " + str(row["B"])

print(formatted)

print("\nQ2 Answer")
print(len(formatted))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.

Q2 Answer
407


### Question - 3 

In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

row = train.iloc[0]

choices = [

    str(row["prompt"]) + " [SEP] " + str(row[c])

    for c in ["A","B","C","D","E"]

]

encoding = tokenizer(

    choices,

    padding="max_length",

    truncation=True,

    max_length=128,

    return_tensors="pt"

)

input_ids = encoding["input_ids"].unsqueeze(0)

print(input_ids.shape)

print("\nQ3 Answer")

print(input_ids.shape[1])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

torch.Size([1, 5, 128])

Q3 Answer
5


### Question - 4 

In [6]:
batch_inputs = []

for i in range(16):

    row = train.iloc[i]

    choices = [

        str(row["prompt"])+" [SEP] "+str(row[c])

        for c in ["A","B","C","D","E"]

    ]

    enc = tokenizer(

        choices,

        padding="max_length",

        truncation=True,

        max_length=128,

        return_tensors="pt"

    )

    batch_inputs.append(enc["input_ids"])

batch_inputs = torch.stack(batch_inputs)

print(batch_inputs.shape)

print("\nQ4 Answer")

print(batch_inputs.numel())

torch.Size([16, 5, 128])

Q4 Answer
10240


### Question - 5

In [7]:
model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

row = train.iloc[0]

choices = [

    str(row["prompt"])+" [SEP] "+str(row[c])

    for c in ["A","B","C","D","E"]

]

encoding = tokenizer(

    choices,

    padding="max_length",

    truncation=True,

    max_length=128,

    return_tensors="pt"

)

inputs = {

    "input_ids":encoding["input_ids"].unsqueeze(0),

    "attention_mask":encoding["attention_mask"].unsqueeze(0),

    "labels":torch.tensor([LABEL2ID[row["answer"]]])

}

outputs = model(**inputs)

print(outputs.logits.shape)

print("\nQ5 Answer")

print(outputs.logits.shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 5])

Q5 Answer
5


### Question - 6

In [8]:

print("\nQ6 Answer")

print(outputs.loss.dim())


Q6 Answer
0


### Question - 7

In [9]:
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)

base_model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

lora_config = LoraConfig(

    r=8,

    lora_alpha=16,

    target_modules=[
        "query",
        "value"
    ],

    lora_dropout=0.1,

    bias="none",

    task_type=TaskType.SEQ_CLS

)

model = get_peft_model(
    base_model,
    lora_config
)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Q7 Answer")
print(trainable)

model.print_trainable_parameters()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Q7 Answer
295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


### Question - 8

In [10]:
from datasets import Dataset

small_train = train.iloc[:100].copy()

dataset = Dataset.from_pandas(
    small_train
)

def preprocess(example):

    choices = [

        str(example["prompt"]) + " [SEP] " + str(example[c])

        for c in ["A","B","C","D","E"]

    ]

    tokenized = tokenizer(

        choices,

        padding="max_length",

        truncation=True,

        max_length=128

    )

    tokenized["labels"] = LABEL2ID[
        example["answer"]
    ]

    return tokenized

dataset = dataset.map(
    preprocess
)

print(dataset)

print("\nQ8 Answer")

print(len(dataset[0]["input_ids"]))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

Q8 Answer
5


### Question - 9

In [11]:
tiny_dataset = Dataset.from_pandas(
    train.iloc[:32].copy()
)

tiny_dataset = tiny_dataset.map(
    preprocess
)

training_args = TrainingArguments(

    output_dir="./outputs",

    per_device_train_batch_size=4,

    gradient_accumulation_steps=1,

    max_steps=4,

    logging_steps=1,

    save_strategy="no",

    report_to="none"

)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tiny_dataset,

)

trainer.train()

print("\nQ9 Answer")

print(trainer.state.global_step)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.598874
2,1.565812
3,1.553491
4,1.697172



Q9 Answer
4


### Question - 10

In [12]:
import torch.nn.functional as F

model.eval()

row = train.iloc[0]

choices = [

    str(row["prompt"]) + " [SEP] " + str(row[c])

    for c in ["A","B","C","D","E"]

]

encoding = tokenizer(

    choices,

    padding="max_length",

    truncation=True,

    max_length=64,

    return_tensors="pt"

)

with torch.no_grad():

    outputs = model(

        input_ids=encoding["input_ids"].unsqueeze(0),

        attention_mask=encoding["attention_mask"].unsqueeze(0)

    )

probs = F.softmax(
    outputs.logits,
    dim=1
)

print("Probabilities")

print(probs)

print("\nQ10 Answer")

print(round(
    probs[0][4].item(),
    4
))

Probabilities
tensor([[0.1927, 0.1980, 0.2010, 0.2037, 0.2046]])

Q10 Answer
0.2046
